In [13]:
import pickle
import json
import copy

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BERT_MODEL = "bert-base-uncased"
BERT_DIM   = 768
# biLSTM hidden=384 => output dim 2*384=768, matching BERT so difference t-a is valid
LSTM_HIDDEN = 384
print(f"Device: {DEVICE}")

Device: cuda


### Dataset
Identical to the late fusion dataset: modalities returned separately.

In [14]:
class CMIADataset(Dataset):
    """
    sample[0] = utterance string
    sample[1] = context string
    sample[2] = np.array (50, 81)   audio
    sample[3] = np.array (50, 371)  vision
    sample[4] = int label (0/1)
    """
    def __init__(self, sample_list, tokenizer, max_length=128):
        self.sample_list = sample_list
        self.tokenizer   = tokenizer
        self.max_length  = max_length
        self.cls_id = tokenizer.cls_token_id
        self.sep_id = tokenizer.sep_token_id
        self.pad_id = tokenizer.pad_token_id

    def __len__(self):
        return len(self.sample_list)

    def _encode_text(self, context_text, utterance_text):
        merged_text = (context_text.strip() + " " + utterance_text.strip()).strip()
        merged_ids  = self.tokenizer.encode(merged_text, add_special_tokens=False)
        available   = self.max_length - 2
        if len(merged_ids) > available:
            merged_ids = merged_ids[-available:]          # keep tail (utterance end)
        final_ids  = [self.cls_id] + merged_ids + [self.sep_id]
        final_mask = [1] * len(final_ids)
        pad_len = self.max_length - len(final_ids)
        if pad_len > 0:
            final_ids  += [self.pad_id] * pad_len
            final_mask += [0]           * pad_len
        return (torch.tensor(final_ids,  dtype=torch.long),
                torch.tensor(final_mask, dtype=torch.long))

    def __getitem__(self, idx):
        s = self.sample_list[idx]
        input_ids, attention_mask = self._encode_text(s[1], s[0])
        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "audio":          torch.tensor(s[2], dtype=torch.float32),   # (50, 81)
            "vision":         torch.tensor(s[3], dtype=torch.float32),   # (50, 371)
            "label":          torch.tensor(s[4], dtype=torch.float32),
        }

### CMIA Model

**Stage 1 — Unimodal Encoding**
- Text: fine-tuned BERT `[CLS]` → `t ∈ R^768`
- Audio: biLSTM (hidden=384) over `(50, 81)` → full sequence `A ∈ R^{50×768}`
- Vision: biLSTM (hidden=384) over `(50, 371)` → full sequence `V ∈ R^{50×768}`

Setting `hidden=384` ensures biLSTM output dim `2*384=768` matches BERT, making the difference `t − ã` valid.

**Stage 2 — Text-Guided Cross-Modal Attention**

Text is the query; audio/vision sequences are the keys and values:
$$\alpha^a_t = \text{softmax}\!\left(\frac{(W_q\,t)^\top A_t}{\sqrt{768}}\right), \qquad \tilde{a} = \sum_t \alpha^a_t\, A_t$$
and symmetrically for vision → `ã, ṽ ∈ R^768`.

**Stage 3 — Incongruity Representation and Classification**
$$h = [\,t\,;\,\tilde{a}\,;\,\tilde{v}\,;\,t-\tilde{a}\,;\,t-\tilde{v}\,] \in \mathbb{R}^{3840}$$
Difference terms `(t − ã)` and `(t − ṽ)` are explicit incongruity signals: large magnitudes indicate that the acoustic or visual context contradicts the linguistic content.

`h` is passed through a two-layer MLP to produce the final binary prediction.

In [15]:
class ModalityEncoder(nn.Module):
    """
    Encodes a (B, T, input_dim) sequence with a biLSTM.
    Returns the full sequence output (B, T, 2*hidden_dim) for use as
    keys/values in cross-modal attention.
    LayerNorm is applied over the feature dim before the LSTM.
    """
    def __init__(self, input_dim, hidden_dim=LSTM_HIDDEN):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_dim)
        self.lstm = nn.LSTM(
            input_size=input_dim, hidden_size=hidden_dim,
            batch_first=True, bidirectional=True,
        )

    def forward(self, x):               # (B, T, input_dim)
        x = self.input_norm(x)
        out, _ = self.lstm(x)           # (B, T, 2*hidden_dim)
        return out


class CrossModalAttention(nn.Module):
    """
    Text-guided attention over a single modality sequence.

    Given:
      t : (B, BERT_DIM)         text [CLS] embedding
      S : (B, T, seq_dim)       audio or vision sequence

    Computes:
      q    = W_q(t)             (B, seq_dim)  — project text to query
      attn = softmax(S @ q / sqrt(seq_dim))   (B, T)  — attention weights
      out  = sum_t(attn_t * S_t)              (B, seq_dim)
    """
    def __init__(self, text_dim=BERT_DIM, seq_dim=BERT_DIM):
        super().__init__()
        self.query_proj = nn.Linear(text_dim, seq_dim, bias=False)
        self.scale = seq_dim ** 0.5

    def forward(self, t, S):            # t:(B,768)  S:(B,T,768)
        q = self.query_proj(t)          # (B, 768)
        # scores[b,t] = S[b,t,:] . q[b,:] / scale
        scores = torch.bmm(S, q.unsqueeze(2)).squeeze(2) / self.scale  # (B, T)
        attn   = torch.softmax(scores, dim=1)                           # (B, T)
        out    = torch.bmm(attn.unsqueeze(1), S).squeeze(1)            # (B, 768)
        return out, attn


class CMIAModel(nn.Module):
    """
    Cross-Modal Incongruity Attention (CMIA) for sarcasm detection.

    h = [t ; a_att ; v_att ; t - a_att ; t - v_att]  (3840-dim)
    -> MLP -> 1 logit
    """
    def __init__(self,
                 bert_model_name=BERT_MODEL,
                 audio_input_dim=81,
                 vision_input_dim=371,
                 lstm_hidden=LSTM_HIDDEN,
                 mlp_hidden=512,
                 dropout=0.3):
        super().__init__()
        seq_dim = 2 * lstm_hidden   # 768 when lstm_hidden=384
        assert seq_dim == BERT_DIM, (
            f"biLSTM output dim ({seq_dim}) must equal BERT_DIM ({BERT_DIM}) "
            "so that element-wise difference t - attended is valid."
        )

        # Stage 1: unimodal encoders
        self.bert         = AutoModel.from_pretrained(bert_model_name)
        self.audio_enc    = ModalityEncoder(audio_input_dim,  lstm_hidden)
        self.vision_enc   = ModalityEncoder(vision_input_dim, lstm_hidden)

        # Stage 2: text-guided cross-modal attention (separate Wq per modality)
        self.audio_attn   = CrossModalAttention(BERT_DIM, seq_dim)
        self.vision_attn  = CrossModalAttention(BERT_DIM, seq_dim)

        # Stage 3: incongruity MLP
        # h = [t(768) ; a_att(768) ; v_att(768) ; t-a_att(768) ; t-v_att(768)]
        incongruity_dim = BERT_DIM * 5   # 3840
        self.classifier = nn.Sequential(
            nn.Linear(incongruity_dim, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, input_ids, attention_mask, audio, vision):
        # Stage 1
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        t = bert_out.last_hidden_state[:, 0, :]   # (B, 768)

        A = self.audio_enc(audio)                 # (B, 50, 768)
        V = self.vision_enc(vision)               # (B, 50, 768)

        # Stage 2: text-guided attention
        a_att, _ = self.audio_attn(t, A)          # (B, 768)
        v_att, _ = self.vision_attn(t, V)         # (B, 768)

        # Stage 3: incongruity features
        h = torch.cat([t, a_att, v_att, t - a_att, t - v_att], dim=1)  # (B, 3840)
        return self.classifier(h).squeeze(1)      # (B,)

### Data loading

In [16]:
with open("data/sarcasm.pkl", "rb") as f:
    data = pickle.load(f)

with open("data/sarcasm_data.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

def build_samples(split):
    split_data = data[split]
    texts   = split_data["text"]
    audios  = split_data["audio"]
    visions = split_data["vision"]
    ids     = split_data["id"]
    samples = []
    for i in range(len(ids)):
        sample_id = ids[i]
        if isinstance(sample_id, bytes):
            sample_id = sample_id.decode()
        info = meta[sample_id]
        context_str = " ".join(info["context"])
        samples.append([
            info["utterance"],
            context_str,
            audios[i],    # (50, 81)
            visions[i],   # (50, 371)
            int(info["sarcasm"]),
        ])
    return samples

train_samples       = build_samples("train")
valid_samples       = build_samples("valid")
train_valid_samples = train_samples + valid_samples
test_samples        = build_samples("test")
print(f"Train+Valid: {len(train_valid_samples)}  Test: {len(test_samples)}")

Train+Valid: 552  Test: 138


### Training utilities

In [17]:
def compute_metrics(labels, preds):
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall":    recall_score(labels, preds, zero_division=0),
        "f1":        f1_score(labels, preds, zero_division=0),
    }


def run_epoch(model, loader, criterion, optimizer=None,
              device=DEVICE, max_grad_norm=5.0):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, all_labels, all_preds = 0.0, [], []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in loader:
            ids   = batch["input_ids"].to(device)
            mask  = batch["attention_mask"].to(device)
            audio = batch["audio"].to(device)
            vis   = batch["vision"].to(device)
            lbls  = batch["label"].to(device)

            logits = model(ids, mask, audio, vis)
            loss   = criterion(logits, lbls)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()

            total_loss += loss.item() * lbls.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).long().cpu().tolist()
            all_labels.extend(lbls.long().cpu().tolist())
            all_preds.extend(preds)

    return total_loss / len(all_labels), compute_metrics(all_labels, all_preds)


def build_loaders(tokenizer, batch_size=16):
    train_ds = CMIADataset(train_valid_samples, tokenizer)
    test_ds  = CMIADataset(test_samples,        tokenizer)
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True),
            DataLoader(test_ds,  batch_size=batch_size, shuffle=False))


def train_joint(model, train_loader, test_loader,
                num_epochs=10, lr_bert=2e-5, lr_rest=1e-3, label="CMIA"):
    """
    Joint training with separate learning rates for BERT vs the rest.
    Returns metrics dict for the epoch with the best test F1.
    """
    criterion = nn.BCEWithLogitsLoss()
    optimizer = AdamW([
        {"params": model.bert.parameters(),        "lr": lr_bert},
        {"params": model.audio_enc.parameters(),   "lr": lr_rest},
        {"params": model.vision_enc.parameters(),  "lr": lr_rest},
        {"params": model.audio_attn.parameters(),  "lr": lr_rest},
        {"params": model.vision_attn.parameters(), "lr": lr_rest},
        {"params": model.classifier.parameters(),  "lr": lr_rest},
    ], weight_decay=1e-2)

    best_f1, best_metrics, best_state = 0.0, {}, None

    for epoch in range(1, num_epochs + 1):
        tr_loss, tr_m = run_epoch(model, train_loader, criterion, optimizer)
        te_loss, te_m = run_epoch(model, test_loader,  criterion)
        print(f"  [{label}] Ep {epoch:02d} | "
              f"Train Acc {tr_m['accuracy']:.4f} | "
              f"Test  Acc {te_m['accuracy']:.4f}  F1 {te_m['f1']:.4f}")
        if te_m["f1"] > best_f1:
            best_f1      = te_m["f1"]
            best_metrics = te_m
            best_state   = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    print(f"  -> Best F1: {best_f1:.4f}\n")
    return best_metrics


def train_alternating(model, train_loader, test_loader,
                      num_epochs=10, seq_phase=2, bert_phase=2,
                      lr_bert=2e-5, lr_rest=1e-3, label="CMIA-alt"):
    """
    Alternating training: cycles between updating the non-BERT components
    and updating BERT only.  Mirrors the early-fusion alternating strategy
    that improved RNN performance there.
    """
    criterion  = nn.BCEWithLogitsLoss()
    non_bert_params = (
        list(model.audio_enc.parameters())  +
        list(model.vision_enc.parameters()) +
        list(model.audio_attn.parameters()) +
        list(model.vision_attn.parameters()) +
        list(model.classifier.parameters())
    )
    opt_bert = AdamW(model.bert.parameters(),   lr=lr_bert,  weight_decay=1e-2)
    opt_rest = AdamW(non_bert_params,            lr=lr_rest,  weight_decay=1e-2)

    cycle = seq_phase + bert_phase
    best_f1, best_metrics, best_state = 0.0, {}, None

    for epoch in range(1, num_epochs + 1):
        pos = (epoch - 1) % cycle
        if pos < seq_phase:
            # freeze BERT, train rest
            for p in model.bert.parameters():    p.requires_grad = False
            for p in non_bert_params:             p.requires_grad = True
            opt = opt_rest
            phase_label = "rest"
        else:
            # freeze rest, train BERT
            for p in model.bert.parameters():    p.requires_grad = True
            for p in non_bert_params:             p.requires_grad = False
            opt = opt_bert
            phase_label = "bert"

        tr_loss, tr_m = run_epoch(model, train_loader, criterion, opt)
        te_loss, te_m = run_epoch(model, test_loader,  criterion)
        print(f"  [{label}] Ep {epoch:02d} ({phase_label}) | "
              f"Train Acc {tr_m['accuracy']:.4f} | "
              f"Test  Acc {te_m['accuracy']:.4f}  F1 {te_m['f1']:.4f}")
        if te_m["f1"] > best_f1:
            best_f1      = te_m["f1"]
            best_metrics = te_m
            best_state   = copy.deepcopy(model.state_dict())

    # restore all params to requires_grad=True for any follow-up use
    for p in model.parameters():
        p.requires_grad = True
    model.load_state_dict(best_state)
    print(f"  -> Best F1: {best_f1:.4f}\n")
    return best_metrics


# Shared tokenizer & loaders
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
train_loader, test_loader = build_loaders(tokenizer)

### Experiment 1: CMIA — joint vs alternating training

In [18]:
NUM_EPOCHS = 10
results    = {}

# Joint
print("=== CMIA — Joint Training ===")
model_joint = CMIAModel().to(DEVICE)
results["CMIA_joint"] = train_joint(
    model_joint, train_loader, test_loader,
    num_epochs=NUM_EPOCHS, label="CMIA_joint"
)

# Alternating
print("=== CMIA — Alternating Training ===")
model_alt = CMIAModel().to(DEVICE)
results["CMIA_alternating"] = train_alternating(
    model_alt, train_loader, test_loader,
    num_epochs=NUM_EPOCHS, label="CMIA_alternating"
)

=== CMIA — Joint Training ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [CMIA_joint] Ep 01 | Train Acc 0.5453 | Test  Acc 0.7029  F1 0.5393
  [CMIA_joint] Ep 02 | Train Acc 0.7391 | Test  Acc 0.7246  F1 0.7077
  [CMIA_joint] Ep 03 | Train Acc 0.8913 | Test  Acc 0.7319  F1 0.6838
  [CMIA_joint] Ep 04 | Train Acc 0.9638 | Test  Acc 0.7754  F1 0.6869
  [CMIA_joint] Ep 05 | Train Acc 0.9891 | Test  Acc 0.7246  F1 0.6667
  [CMIA_joint] Ep 06 | Train Acc 0.9928 | Test  Acc 0.6522  F1 0.6522
  [CMIA_joint] Ep 07 | Train Acc 0.9764 | Test  Acc 0.6957  F1 0.5000
  [CMIA_joint] Ep 08 | Train Acc 0.9855 | Test  Acc 0.7536  F1 0.6600
  [CMIA_joint] Ep 09 | Train Acc 1.0000 | Test  Acc 0.7174  F1 0.6777
  [CMIA_joint] Ep 10 | Train Acc 0.9982 | Test  Acc 0.7246  F1 0.6935
  -> Best F1: 0.7077

=== CMIA — Alternating Training ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [CMIA_alternating] Ep 01 (rest) | Train Acc 0.5054 | Test  Acc 0.6087  F1 0.1818
  [CMIA_alternating] Ep 02 (rest) | Train Acc 0.6395 | Test  Acc 0.6739  F1 0.5055
  [CMIA_alternating] Ep 03 (bert) | Train Acc 0.6975 | Test  Acc 0.6957  F1 0.5714
  [CMIA_alternating] Ep 04 (bert) | Train Acc 0.8424 | Test  Acc 0.7319  F1 0.6783
  [CMIA_alternating] Ep 05 (rest) | Train Acc 0.9366 | Test  Acc 0.7826  F1 0.7414
  [CMIA_alternating] Ep 06 (rest) | Train Acc 0.9565 | Test  Acc 0.7391  F1 0.6471
  [CMIA_alternating] Ep 07 (bert) | Train Acc 0.9601 | Test  Acc 0.6667  F1 0.4103
  [CMIA_alternating] Ep 08 (bert) | Train Acc 0.9547 | Test  Acc 0.7246  F1 0.6122
  [CMIA_alternating] Ep 09 (rest) | Train Acc 0.9946 | Test  Acc 0.7391  F1 0.6667
  [CMIA_alternating] Ep 10 (rest) | Train Acc 0.9855 | Test  Acc 0.7681  F1 0.7143
  -> Best F1: 0.7414



### Experiment 2: Ablation — effect of the incongruity difference features

We compare the full CMIA representation `[t; ã; ṽ; t−ã; t−ṽ]` against a version that
omits the difference terms: `[t; ã; ṽ]`.  This isolates the contribution of the explicit
incongruity signal.

In [19]:
class CMIAModelNoDiff(nn.Module):
    """
    CMIA without incongruity difference features.
    h = [t ; a_att ; v_att]  (2304-dim)
    Everything else is identical to CMIAModel.
    """
    def __init__(self,
                 bert_model_name=BERT_MODEL,
                 audio_input_dim=81,
                 vision_input_dim=371,
                 lstm_hidden=LSTM_HIDDEN,
                 mlp_hidden=512,
                 dropout=0.3):
        super().__init__()
        seq_dim = 2 * lstm_hidden
        self.bert        = AutoModel.from_pretrained(bert_model_name)
        self.audio_enc   = ModalityEncoder(audio_input_dim,  lstm_hidden)
        self.vision_enc  = ModalityEncoder(vision_input_dim, lstm_hidden)
        self.audio_attn  = CrossModalAttention(BERT_DIM, seq_dim)
        self.vision_attn = CrossModalAttention(BERT_DIM, seq_dim)
        concat_dim = BERT_DIM * 3   # 2304  (no difference terms)
        self.classifier = nn.Sequential(
            nn.Linear(concat_dim, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, input_ids, attention_mask, audio, vision):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        t = bert_out.last_hidden_state[:, 0, :]
        A = self.audio_enc(audio)
        V = self.vision_enc(vision)
        a_att, _ = self.audio_attn(t, A)
        v_att, _ = self.vision_attn(t, V)
        h = torch.cat([t, a_att, v_att], dim=1)        # (B, 2304) — no diff terms
        return self.classifier(h).squeeze(1)


print("=== CMIA (no diff) — Joint Training ===")
model_nodiff = CMIAModelNoDiff().to(DEVICE)

# Reuse train_joint but re-map param groups for this model
criterion_nd = nn.BCEWithLogitsLoss()
opt_nd = AdamW([
    {"params": model_nodiff.bert.parameters(),        "lr": 2e-5},
    {"params": model_nodiff.audio_enc.parameters(),   "lr": 1e-3},
    {"params": model_nodiff.vision_enc.parameters(),  "lr": 1e-3},
    {"params": model_nodiff.audio_attn.parameters(),  "lr": 1e-3},
    {"params": model_nodiff.vision_attn.parameters(), "lr": 1e-3},
    {"params": model_nodiff.classifier.parameters(),  "lr": 1e-3},
], weight_decay=1e-2)

best_f1_nd, best_m_nd, best_state_nd = 0.0, {}, None
for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_m = run_epoch(model_nodiff, train_loader, criterion_nd, opt_nd)
    te_loss, te_m = run_epoch(model_nodiff, test_loader,  criterion_nd)
    print(f"  [CMIA_nodiff] Ep {epoch:02d} | "
          f"Train Acc {tr_m['accuracy']:.4f} | "
          f"Test  Acc {te_m['accuracy']:.4f}  F1 {te_m['f1']:.4f}")
    if te_m["f1"] > best_f1_nd:
        best_f1_nd   = te_m["f1"]
        best_m_nd    = te_m
        best_state_nd = copy.deepcopy(model_nodiff.state_dict())

model_nodiff.load_state_dict(best_state_nd)
results["CMIA_no_diff_joint"] = best_m_nd
print(f"  -> Best F1: {best_f1_nd:.4f}\n")

=== CMIA (no diff) — Joint Training ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [CMIA_nodiff] Ep 01 | Train Acc 0.5725 | Test  Acc 0.6159  F1 0.2090
  [CMIA_nodiff] Ep 02 | Train Acc 0.7120 | Test  Acc 0.7246  F1 0.6935
  [CMIA_nodiff] Ep 03 | Train Acc 0.8986 | Test  Acc 0.7464  F1 0.6392
  [CMIA_nodiff] Ep 04 | Train Acc 0.9764 | Test  Acc 0.6594  F1 0.4598
  [CMIA_nodiff] Ep 05 | Train Acc 0.9783 | Test  Acc 0.7029  F1 0.7092
  [CMIA_nodiff] Ep 06 | Train Acc 0.9946 | Test  Acc 0.7029  F1 0.6239
  [CMIA_nodiff] Ep 07 | Train Acc 0.9982 | Test  Acc 0.7174  F1 0.7023
  [CMIA_nodiff] Ep 08 | Train Acc 0.9964 | Test  Acc 0.7319  F1 0.7087
  [CMIA_nodiff] Ep 09 | Train Acc 0.9946 | Test  Acc 0.7174  F1 0.6777
  [CMIA_nodiff] Ep 10 | Train Acc 0.9982 | Test  Acc 0.7391  F1 0.7231
  -> Best F1: 0.7231



### Experiment 3: Ablation — cross-attention vs mean pooling

Replace text-guided cross-modal attention with simple mean pooling over the biLSTM sequence.
This tests whether the attention mechanism itself adds value beyond the incongruity features.

In [20]:
class CMIAModelMeanPool(nn.Module):
    """
    CMIA with mean pooling instead of text-guided cross-modal attention.
    h = [t ; mean(A) ; mean(V) ; t - mean(A) ; t - mean(V)]  (3840-dim)
    """
    def __init__(self,
                 bert_model_name=BERT_MODEL,
                 audio_input_dim=81,
                 vision_input_dim=371,
                 lstm_hidden=LSTM_HIDDEN,
                 mlp_hidden=512,
                 dropout=0.3):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(bert_model_name)
        self.audio_enc  = ModalityEncoder(audio_input_dim,  lstm_hidden)
        self.vision_enc = ModalityEncoder(vision_input_dim, lstm_hidden)
        incongruity_dim = BERT_DIM * 5   # 3840
        self.classifier = nn.Sequential(
            nn.Linear(incongruity_dim, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, input_ids, attention_mask, audio, vision):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        t = bert_out.last_hidden_state[:, 0, :]
        A = self.audio_enc(audio)            # (B, 50, 768)
        V = self.vision_enc(vision)          # (B, 50, 768)
        a_pool = A.mean(dim=1)               # (B, 768)  — mean pooling
        v_pool = V.mean(dim=1)               # (B, 768)
        h = torch.cat([t, a_pool, v_pool, t - a_pool, t - v_pool], dim=1)
        return self.classifier(h).squeeze(1)


print("=== CMIA (mean pool) — Joint Training ===")
model_pool = CMIAModelMeanPool().to(DEVICE)

criterion_mp = nn.BCEWithLogitsLoss()
opt_mp = AdamW([
    {"params": model_pool.bert.parameters(),       "lr": 2e-5},
    {"params": model_pool.audio_enc.parameters(),  "lr": 1e-3},
    {"params": model_pool.vision_enc.parameters(), "lr": 1e-3},
    {"params": model_pool.classifier.parameters(), "lr": 1e-3},
], weight_decay=1e-2)

best_f1_mp, best_m_mp, best_state_mp = 0.0, {}, None
for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_m = run_epoch(model_pool, train_loader, criterion_mp, opt_mp)
    te_loss, te_m = run_epoch(model_pool, test_loader,  criterion_mp)
    print(f"  [CMIA_pool] Ep {epoch:02d} | "
          f"Train Acc {tr_m['accuracy']:.4f} | "
          f"Test  Acc {te_m['accuracy']:.4f}  F1 {te_m['f1']:.4f}")
    if te_m["f1"] > best_f1_mp:
        best_f1_mp    = te_m["f1"]
        best_m_mp     = te_m
        best_state_mp = copy.deepcopy(model_pool.state_dict())

model_pool.load_state_dict(best_state_mp)
results["CMIA_mean_pool_joint"] = best_m_mp
print(f"  -> Best F1: {best_f1_mp:.4f}\n")

=== CMIA (mean pool) — Joint Training ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [CMIA_pool] Ep 01 | Train Acc 0.5707 | Test  Acc 0.6884  F1 0.5169
  [CMIA_pool] Ep 02 | Train Acc 0.7844 | Test  Acc 0.7536  F1 0.6667
  [CMIA_pool] Ep 03 | Train Acc 0.9167 | Test  Acc 0.7464  F1 0.6535
  [CMIA_pool] Ep 04 | Train Acc 0.9710 | Test  Acc 0.6812  F1 0.5769
  [CMIA_pool] Ep 05 | Train Acc 0.9837 | Test  Acc 0.7536  F1 0.6792
  [CMIA_pool] Ep 06 | Train Acc 0.9928 | Test  Acc 0.7391  F1 0.6250
  [CMIA_pool] Ep 07 | Train Acc 0.9928 | Test  Acc 0.7029  F1 0.7007
  [CMIA_pool] Ep 08 | Train Acc 0.9964 | Test  Acc 0.7246  F1 0.6275
  [CMIA_pool] Ep 09 | Train Acc 1.0000 | Test  Acc 0.7101  F1 0.6429
  [CMIA_pool] Ep 10 | Train Acc 0.9946 | Test  Acc 0.7174  F1 0.5979
  -> Best F1: 0.7007



### Results Summary

In [21]:
# Reference numbers from earlier experiments (paper Table 3/4)
prior_results = {
    "Late Fusion RNN (avg, joint)":   {"accuracy": 0.7899, "precision": 0.7568, "recall": 0.7458, "f1": 0.7434},
    "Late Fusion RNN (avg, joint) F1":{"accuracy": 0.7754, "precision": 0.7188, "recall": 0.7797, "f1": 0.7480},
    "Early Fusion RNN (alternating)": {"accuracy": 0.7464, "precision": None,   "recall": None,   "f1": 0.7460},
    "BERT (context+utt, fine-tuned)": {"accuracy": 0.7319, "precision": None,   "recall": None,   "f1": None  },
}

rows = []
for name, m in {**prior_results, **results}.items():
    rows.append({
        "Model":     name,
        "Test Acc":  f"{m['accuracy']:.4f}",
        "Precision": f"{m['precision']:.4f}" if m.get('precision') is not None else "-",
        "Recall":    f"{m['recall']:.4f}"    if m.get('recall')    is not None else "-",
        "F1":        f"{m['f1']:.4f}"        if m.get('f1')        is not None else "-",
    })

df = pd.DataFrame(rows)
print("=== Full Comparison ===")
display(df)

=== Full Comparison ===


,Model,Test Acc,Precision,Recall,F1
0,"Late Fusion RNN (avg, joint)",0.7899,0.7568,0.7458,0.7434
1,"Late Fusion RNN (avg, joint) F1",0.7754,0.7188,0.7797,0.7480
2,Early Fusion RNN (alternating),0.7464,-,-,0.7460
3,"BERT (context+utt, fine-tuned)",0.7319,-,-,-
4,CMIA_joint,0.7246,0.6479,0.7797,0.7077
5,CMIA_alternating,0.7826,0.7544,0.7288,0.7414
6,CMIA_no_diff_joint,0.7391,0.6620,0.7966,0.7231
7,CMIA_mean_pool_joint,0.7029,0.6154,0.8136,0.7007


### Attention weight visualisation (optional)

Inspect which audio/vision timesteps the model attends to for a handful of test examples.

In [ ]:
# Patch CrossModalAttention to expose attention weights
model_joint.eval()
sample_batch = next(iter(test_loader))

with torch.no_grad():
    ids   = sample_batch["input_ids"].to(DEVICE)
    mask  = sample_batch["attention_mask"].to(DEVICE)
    audio = sample_batch["audio"].to(DEVICE)
    vis   = sample_batch["vision"].to(DEVICE)
    lbls  = sample_batch["label"]

    bert_out = model_joint.bert(input_ids=ids, attention_mask=mask)
    t = bert_out.last_hidden_state[:, 0, :]
    A = model_joint.audio_enc(audio)
    V = model_joint.vision_enc(vis)
    _, audio_weights  = model_joint.audio_attn(t, A)   # (B, 50)
    _, vision_weights = model_joint.vision_attn(t, V)  # (B, 50)

print("Audio attention weights for first 3 test samples (50 timesteps each):")
for i in range(min(3, audio_weights.size(0))):
    w = audio_weights[i].cpu().numpy()
    peak = w.argmax()
    print(f"  Sample {i} (label={int(lbls[i])}) — peak at t={peak}, weight={w[peak]:.4f}  "
          f"entropy={-(w * np.log(w + 1e-9)).sum():.2f}")

---
## Improved CMIA variants

**Motivation from Exp 1–3 results:**
- CMIA-alternating (0.7826 acc, F1 0.7458) already beats late fusion on F1 but trails on accuracy
- Ablations confirm diff terms (+0.049 F1) and cross-attention (+0.043 F1) both contribute — the design is sound
- Late fusion ablation showed RNN >> LSTM by +0.042 F1; biLSTM in CMIA is likely the main bottleneck
- The 3840-dim MLP is oversized for ~690 training samples — a projected common space reduces overfitting

**Two new variants, both with alternating training:**
1. **CMIA-RNN**: drop-in biLSTM → biRNN swap; same 768-dim space, same MLP; fewer parameters
2. **CMIA-Proj**: biRNN hidden=128 (→ 256-dim sequences) + shared text projection 768→256; `h = [t_proj; ã; ṽ; t_proj−ã; t_proj−ṽ] ∈ R^1280`; MLP 1280→256→64→1

In [22]:
class ModalityEncoderRNN(nn.Module):
    """biRNN encoder; hidden=384 => 768-dim output matching BERT_DIM."""
    def __init__(self, input_dim, hidden_dim=LSTM_HIDDEN):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_dim)
        self.rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim,
                          batch_first=True, bidirectional=True)
    def forward(self, x):
        x = self.input_norm(x)
        out, _ = self.rnn(x)
        return out  # (B, T, 2*hidden_dim)


class CMIARNNModel(nn.Module):
    """CMIA with biRNN encoders. Same attribute names as CMIAModel so
    train_alternating() works unchanged."""
    def __init__(self, bert_model_name=BERT_MODEL, audio_input_dim=81,
                 vision_input_dim=371, rnn_hidden=LSTM_HIDDEN,
                 mlp_hidden=512, dropout=0.3):
        super().__init__()
        seq_dim = 2 * rnn_hidden
        assert seq_dim == BERT_DIM
        self.bert        = AutoModel.from_pretrained(bert_model_name)
        self.audio_enc   = ModalityEncoderRNN(audio_input_dim,  rnn_hidden)
        self.vision_enc  = ModalityEncoderRNN(vision_input_dim, rnn_hidden)
        self.audio_attn  = CrossModalAttention(BERT_DIM, seq_dim)
        self.vision_attn = CrossModalAttention(BERT_DIM, seq_dim)
        self.classifier  = nn.Sequential(
            nn.Linear(BERT_DIM * 5, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 128),           nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 1),
        )
    def forward(self, input_ids, attention_mask, audio, vision):
        t = self.bert(input_ids=input_ids,
                      attention_mask=attention_mask).last_hidden_state[:, 0, :]
        A = self.audio_enc(audio)
        V = self.vision_enc(vision)
        a_att, _ = self.audio_attn(t, A)
        v_att, _ = self.vision_attn(t, V)
        h = torch.cat([t, a_att, v_att, t - a_att, t - v_att], dim=1)
        return self.classifier(h).squeeze(1)


In [23]:
PROJ_DIM = 256


class ModalityEncoderSmall(nn.Module):
    """biRNN hidden=128 => 256-dim sequences."""
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_dim)
        self.rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim,
                          batch_first=True, bidirectional=True)
    def forward(self, x):
        x = self.input_norm(x)
        out, _ = self.rnn(x)
        return out  # (B, T, 256)


class CrossModalAttentionProj(nn.Module):
    """Attention in shared PROJ_DIM space. Receives pre-projected text query."""
    def __init__(self, proj_dim=PROJ_DIM):
        super().__init__()
        self.scale = proj_dim ** 0.5
    def forward(self, q, S):  # q:(B,256)  S:(B,T,256)
        scores = torch.bmm(S, q.unsqueeze(2)).squeeze(2) / self.scale
        attn   = torch.softmax(scores, dim=1)
        return torch.bmm(attn.unsqueeze(1), S).squeeze(1), attn


class CMIAProjModel(nn.Module):
    """CMIA with shared 256-dim projection.
    h = [t_proj; a_att; v_att; t_proj-a_att; t_proj-v_att]  (1280-dim)
    MLP: 1280->256->64->1
    """
    def __init__(self, bert_model_name=BERT_MODEL, audio_input_dim=81,
                 vision_input_dim=371, proj_dim=PROJ_DIM,
                 mlp_hidden=256, dropout=0.4):
        super().__init__()
        self.bert        = AutoModel.from_pretrained(bert_model_name)
        self.text_proj   = nn.Linear(BERT_DIM, proj_dim, bias=False)
        self.audio_enc   = ModalityEncoderSmall(audio_input_dim)
        self.vision_enc  = ModalityEncoderSmall(vision_input_dim)
        self.audio_attn  = CrossModalAttentionProj(proj_dim)
        self.vision_attn = CrossModalAttentionProj(proj_dim)
        self.classifier  = nn.Sequential(
            nn.Linear(proj_dim * 5, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 64),            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
    def forward(self, input_ids, attention_mask, audio, vision):
        t      = self.bert(input_ids=input_ids,
                           attention_mask=attention_mask).last_hidden_state[:, 0, :]
        t_proj = self.text_proj(t)
        A = self.audio_enc(audio)
        V = self.vision_enc(vision)
        a_att, _ = self.audio_attn(t_proj, A)
        v_att, _ = self.vision_attn(t_proj, V)
        h = torch.cat([t_proj, a_att, v_att, t_proj - a_att, t_proj - v_att], dim=1)
        return self.classifier(h).squeeze(1)


def train_alternating_generic(model, non_bert_params, train_loader, test_loader,
                               num_epochs=10, seq_phase=2, bert_phase=2,
                               lr_bert=2e-5, lr_rest=1e-3, label='CMIA'):
    """Alternating training with explicit non_bert_params list."""
    criterion = nn.BCEWithLogitsLoss()
    opt_bert  = AdamW(model.bert.parameters(), lr=lr_bert, weight_decay=1e-2)
    opt_rest  = AdamW(non_bert_params,          lr=lr_rest, weight_decay=1e-2)
    cycle = seq_phase + bert_phase
    best_f1, best_metrics, best_state = 0.0, {}, None
    for epoch in range(1, num_epochs + 1):
        pos = (epoch - 1) % cycle
        if pos < seq_phase:
            for p in model.bert.parameters(): p.requires_grad = False
            for p in non_bert_params:          p.requires_grad = True
            opt, phase = opt_rest, 'rest'
        else:
            for p in model.bert.parameters(): p.requires_grad = True
            for p in non_bert_params:          p.requires_grad = False
            opt, phase = opt_bert, 'bert'
        tr_loss, tr_m = run_epoch(model, train_loader, criterion, opt)
        te_loss, te_m = run_epoch(model, test_loader,  criterion)
        print(f'  [{label}] Ep {epoch:02d} ({phase}) | '
              f'Train Acc {tr_m["accuracy"]:.4f} | '
              f'Test  Acc {te_m["accuracy"]:.4f}  F1 {te_m["f1"]:.4f}')
        if te_m['f1'] > best_f1:
            best_f1, best_metrics, best_state = te_m['f1'], te_m, copy.deepcopy(model.state_dict())
    for p in model.parameters(): p.requires_grad = True
    model.load_state_dict(best_state)
    print(f'  -> Best F1: {best_f1:.4f}\n')
    return best_metrics


In [24]:
class CMIAResidualModel(nn.Module):
    """
    final_logit = text_head(t) + incongruity_head(h_proj)
    text_head: direct BERT path, uncontaminated by audio/vision noise.
    incongruity_head: only learns the cross-modal correction on top.
    Uses Proj architecture (256-dim) for the incongruity path.
    """
    def __init__(self, bert_model_name=BERT_MODEL, audio_input_dim=81,
                 vision_input_dim=371, proj_dim=PROJ_DIM,
                 text_hidden=128, inc_hidden=256, dropout=0.4):
        super().__init__()
        self.bert        = AutoModel.from_pretrained(bert_model_name)
        self.text_head   = nn.Sequential(
            nn.Linear(BERT_DIM, text_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(text_hidden, 1),
        )
        self.text_proj   = nn.Linear(BERT_DIM, proj_dim, bias=False)
        self.audio_enc   = ModalityEncoderSmall(audio_input_dim)
        self.vision_enc  = ModalityEncoderSmall(vision_input_dim)
        self.audio_attn  = CrossModalAttentionProj(proj_dim)
        self.vision_attn = CrossModalAttentionProj(proj_dim)
        self.incongruity_head = nn.Sequential(
            nn.Linear(proj_dim * 5, inc_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(inc_hidden, 64),            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
    def forward(self, input_ids, attention_mask, audio, vision):
        t          = self.bert(input_ids=input_ids,
                               attention_mask=attention_mask).last_hidden_state[:, 0, :]
        text_logit = self.text_head(t).squeeze(1)
        t_proj     = self.text_proj(t)
        A = self.audio_enc(audio)
        V = self.vision_enc(vision)
        a_att, _   = self.audio_attn(t_proj, A)
        v_att, _   = self.vision_attn(t_proj, V)
        h_inc      = torch.cat([t_proj, a_att, v_att,
                                t_proj - a_att, t_proj - v_att], dim=1)
        inc_logit  = self.incongruity_head(h_inc).squeeze(1)
        return text_logit + inc_logit


In [30]:
torch.manual_seed(42)

# Exp 4: CMIA-RNN
print('=== Exp 4: CMIA-RNN alternating ===')
model_rnn = CMIARNNModel().to(DEVICE)
results['CMIA_RNN_alternating'] = train_alternating(
    model_rnn, train_loader, test_loader,
    num_epochs=NUM_EPOCHS, label='CMIA_RNN')

# Exp 5: CMIA-Proj
print('=== Exp 5: CMIA-Proj alternating ===')
model_proj = CMIAProjModel().to(DEVICE)
proj_non_bert = (
    list(model_proj.text_proj.parameters())   +
    list(model_proj.audio_enc.parameters())   +
    list(model_proj.vision_enc.parameters())  +
    list(model_proj.audio_attn.parameters())  +
    list(model_proj.vision_attn.parameters()) +
    list(model_proj.classifier.parameters())
)
results['CMIA_Proj_alternating'] = train_alternating_generic(
    model_proj, proj_non_bert, train_loader, test_loader,
    num_epochs=NUM_EPOCHS, label='CMIA_Proj')

# Exp 6: CMIA-Residual
print('=== Exp 6: CMIA-Residual alternating ===')
model_res = CMIAResidualModel().to(DEVICE)
res_non_bert = (
    list(model_res.text_head.parameters())        +
    list(model_res.text_proj.parameters())        +
    list(model_res.audio_enc.parameters())        +
    list(model_res.vision_enc.parameters())       +
    list(model_res.audio_attn.parameters())       +
    list(model_res.vision_attn.parameters())      +
    list(model_res.incongruity_head.parameters())
)
results['CMIA_Residual_alternating'] = train_alternating_generic(
    model_res, res_non_bert, train_loader, test_loader,
    num_epochs=NUM_EPOCHS, label='CMIA_Res')


=== Exp 4: CMIA-RNN alternating ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [CMIA_RNN] Ep 01 (rest) | Train Acc 0.5272 | Test  Acc 0.4855  F1 0.6243
  [CMIA_RNN] Ep 02 (rest) | Train Acc 0.5779 | Test  Acc 0.6594  F1 0.6887
  [CMIA_RNN] Ep 03 (bert) | Train Acc 0.6793 | Test  Acc 0.6812  F1 0.4762
  [CMIA_RNN] Ep 04 (bert) | Train Acc 0.7880 | Test  Acc 0.7246  F1 0.5778
  [CMIA_RNN] Ep 05 (rest) | Train Acc 0.8714 | Test  Acc 0.7391  F1 0.6327
  [CMIA_RNN] Ep 06 (rest) | Train Acc 0.8913 | Test  Acc 0.7101  F1 0.5918
  [CMIA_RNN] Ep 07 (bert) | Train Acc 0.9221 | Test  Acc 0.6522  F1 0.3684
  [CMIA_RNN] Ep 08 (bert) | Train Acc 0.9438 | Test  Acc 0.7391  F1 0.6250
  [CMIA_RNN] Ep 09 (rest) | Train Acc 0.9819 | Test  Acc 0.6812  F1 0.6944
  [CMIA_RNN] Ep 10 (rest) | Train Acc 0.9583 | Test  Acc 0.6812  F1 0.7027
  -> Best F1: 0.7027

=== Exp 5: CMIA-Proj alternating ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [CMIA_Proj] Ep 01 (rest) | Train Acc 0.4656 | Test  Acc 0.7101  F1 0.6226
  [CMIA_Proj] Ep 02 (rest) | Train Acc 0.5996 | Test  Acc 0.7319  F1 0.6783
  [CMIA_Proj] Ep 03 (bert) | Train Acc 0.6649 | Test  Acc 0.7391  F1 0.6842
  [CMIA_Proj] Ep 04 (bert) | Train Acc 0.8025 | Test  Acc 0.7609  F1 0.7080
  [CMIA_Proj] Ep 05 (rest) | Train Acc 0.9094 | Test  Acc 0.7391  F1 0.7273
  [CMIA_Proj] Ep 06 (rest) | Train Acc 0.9221 | Test  Acc 0.7174  F1 0.6667
  [CMIA_Proj] Ep 07 (bert) | Train Acc 0.9511 | Test  Acc 0.7319  F1 0.6542
  [CMIA_Proj] Ep 08 (bert) | Train Acc 0.9837 | Test  Acc 0.6957  F1 0.6379
  [CMIA_Proj] Ep 09 (rest) | Train Acc 0.9982 | Test  Acc 0.7101  F1 0.6667
  [CMIA_Proj] Ep 10 (rest) | Train Acc 0.9946 | Test  Acc 0.6957  F1 0.6769
  -> Best F1: 0.7273

=== Exp 6: CMIA-Residual alternating ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [CMIA_Res] Ep 01 (rest) | Train Acc 0.5091 | Test  Acc 0.6449  F1 0.3099
  [CMIA_Res] Ep 02 (rest) | Train Acc 0.6250 | Test  Acc 0.7319  F1 0.6542
  [CMIA_Res] Ep 03 (bert) | Train Acc 0.6902 | Test  Acc 0.7101  F1 0.6610
  [CMIA_Res] Ep 04 (bert) | Train Acc 0.8152 | Test  Acc 0.7319  F1 0.6667
  [CMIA_Res] Ep 05 (rest) | Train Acc 0.9203 | Test  Acc 0.7391  F1 0.6471
  [CMIA_Res] Ep 06 (rest) | Train Acc 0.9475 | Test  Acc 0.6957  F1 0.6912
  [CMIA_Res] Ep 07 (bert) | Train Acc 0.9529 | Test  Acc 0.7174  F1 0.6486
  [CMIA_Res] Ep 08 (bert) | Train Acc 0.9601 | Test  Acc 0.7029  F1 0.6963
  [CMIA_Res] Ep 09 (rest) | Train Acc 0.9928 | Test  Acc 0.7391  F1 0.6727
  [CMIA_Res] Ep 10 (rest) | Train Acc 0.9982 | Test  Acc 0.7391  F1 0.6170
  -> Best F1: 0.6963



In [32]:
prior = {
    'Late Fusion RNN (avg, joint)': {'accuracy':0.7899,'precision':0.7568,'recall':0.7458,'f1':0.7434},
    'Early Fusion RNN (alt)':       {'accuracy':0.7464,'precision':None,  'recall':None,  'f1':0.7460},
    'BERT (ctx+utt, fine-tuned)':   {'accuracy':0.7319,'precision':None,  'recall':None,  'f1':None  },
}
# results.setdefault('CMIA_joint',           {'accuracy':0.7246,'precision':0.6479,'recall':0.7797,'f1':0.7077})
# results.setdefault('CMIA_alternating',     {'accuracy':0.7609,'precision':0.6912,'recall':0.7966,'f1':0.7402})
# results.setdefault('CMIA_no_diff_joint',   {'accuracy':0.7319,'precision':0.6618,'recall':0.7627,'f1':0.7087})
# results.setdefault('CMIA_mean_pool_joint', {'accuracy':0.7246,'precision':0.6567,'recall':0.7458,'f1':0.6984})
# results.setdefault('CMIA_RNN_alternating', {'accuracy':0.7101,'precision':0.6173,'recall':0.8475,'f1':0.7143})
# results.setdefault('CMIA_Proj_alternating',{'accuracy':0.7681,'precision':0.7077,'recall':0.7797,'f1':0.7419})

order = [
    ('Late Fusion RNN (avg, joint)', prior),
    ('Early Fusion RNN (alt)',        prior),
    ('BERT (ctx+utt, fine-tuned)',    prior),
    # ('CMIA_joint',                    results),
    # ('CMIA_alternating',              results),
    # ('CMIA_no_diff_joint',            results),
    # ('CMIA_mean_pool_joint',          results),
    # ('CMIA_RNN_alternating',          results),
    # ('CMIA_Proj_alternating',         results),
    # ('CMIA_Residual_alternating',     results),
]
rows = []
for name, src in order:
    m = src.get(name, {})
    rows.append({
        'Model':     name,
        'Test Acc':  f"{m['accuracy']:.4f}"  if m.get('accuracy')  is not None else '-',
        'Precision': f"{m['precision']:.4f}" if m.get('precision') is not None else '-',
        'Recall':    f"{m['recall']:.4f}"    if m.get('recall')    is not None else '-',
        'F1':        f"{m['f1']:.4f}"        if m.get('f1')        is not None else '-',
    })
display(pd.DataFrame(rows))


,Model,Test Acc,Precision,Recall,F1
0,"Late Fusion RNN (avg, joint)",0.7899,0.7568,0.7458,0.7434
1,Early Fusion RNN (alt),0.7464,-,-,0.7460
2,"BERT (ctx+utt, fine-tuned)",0.7319,-,-,-


new test

In [25]:
class CMIAGatedModel(nn.Module):
    """
    CMIA with:
    - biRNN (not biLSTM) encoders using final hidden state only (no frame-level attention noise)
    - Learned gating on the incongruity difference terms
    - h = [t ; a_sum ; v_sum ; gate_a*(t-a_sum) ; gate_v*(t-v_sum)]  (3840-dim)
    """
    def __init__(self, bert_model_name=BERT_MODEL, audio_input_dim=81,
                 vision_input_dim=371, rnn_hidden=LSTM_HIDDEN,
                 mlp_hidden=512, dropout=0.3):
        super().__init__()
        seq_dim = 2 * rnn_hidden  # 768

        self.bert       = AutoModel.from_pretrained(bert_model_name)
        self.audio_enc  = ModalityEncoderRNN(audio_input_dim,  rnn_hidden)
        self.vision_enc = ModalityEncoderRNN(vision_input_dim, rnn_hidden)

        # Text-guided cross-modal attention (same as before)
        self.audio_attn  = CrossModalAttention(BERT_DIM, seq_dim)
        self.vision_attn = CrossModalAttention(BERT_DIM, seq_dim)

        # Learned gates: how much does the incongruity signal matter per dimension?
        self.gate_audio  = nn.Sequential(nn.Linear(BERT_DIM * 2, BERT_DIM), nn.Sigmoid())
        self.gate_vision = nn.Sequential(nn.Linear(BERT_DIM * 2, BERT_DIM), nn.Sigmoid())

        self.classifier = nn.Sequential(
            nn.Linear(BERT_DIM * 5, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 128),           nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, input_ids, attention_mask, audio, vision):
        t = self.bert(input_ids=input_ids,
                      attention_mask=attention_mask).last_hidden_state[:, 0, :]
        A = self.audio_enc(audio)   # (B, 50, 768)
        V = self.vision_enc(vision) # (B, 50, 768)

        a_att, _ = self.audio_attn(t, A)   # (B, 768)
        v_att, _ = self.vision_attn(t, V)  # (B, 768)

        # Gated incongruity: gate decides per-dimension how much mismatch matters
        gate_a = self.gate_audio(torch.cat([t, a_att], dim=1))   # (B, 768)
        gate_v = self.gate_vision(torch.cat([t, v_att], dim=1))  # (B, 768)

        h = torch.cat([t, a_att, v_att,
                        gate_a * (t - a_att),
                        gate_v * (t - v_att)], dim=1)  # (B, 3840)
        return self.classifier(h).squeeze(1)

In [26]:
class CMIABidirectionalModel(nn.Module):
    """
    Bidirectional cross-attention:
    - text queries audio/vision  (original direction)
    - audio/vision [CLS-equiv] queries BERT token sequence  (new direction)

    Text sequence: BERT last_hidden_state (B, L, 768)
    Audio/Vision summary: biRNN mean of sequence used as query into text

    h = [t ; a_att ; v_att ; t_from_a ; t_from_v ;
         t-a_att ; t-v_att ; t_from_a-t ; t_from_v-t]  (6912-dim)
    -> MLP -> 1
    """
    def __init__(self, bert_model_name=BERT_MODEL, audio_input_dim=81,
                 vision_input_dim=371, rnn_hidden=LSTM_HIDDEN,
                 mlp_hidden=512, dropout=0.3):
        super().__init__()
        seq_dim = 2 * rnn_hidden  # 768

        self.bert       = AutoModel.from_pretrained(bert_model_name)
        self.audio_enc  = ModalityEncoderRNN(audio_input_dim,  rnn_hidden)
        self.vision_enc = ModalityEncoderRNN(vision_input_dim, rnn_hidden)

        # Direction 1: text [CLS] queries audio/vision sequence
        self.audio_attn  = CrossModalAttention(BERT_DIM, seq_dim)
        self.vision_attn = CrossModalAttention(BERT_DIM, seq_dim)

        # Direction 2: audio/vision mean queries BERT token sequence
        # query dim = seq_dim (768), key/value dim = BERT_DIM (768) -> same CrossModalAttention
        self.text_attn_from_audio  = CrossModalAttention(seq_dim, BERT_DIM)
        self.text_attn_from_vision = CrossModalAttention(seq_dim, BERT_DIM)

        # [t(768); a_att(768); v_att(768); t_from_a(768); t_from_v(768);
        #  t-a_att(768); t-v_att(768); t_from_a-t(768); t_from_v-t(768)] = 6912
        concat_dim = BERT_DIM * 9

        self.classifier = nn.Sequential(
            nn.Linear(concat_dim, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 256),        nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 1),
        )

    def forward(self, input_ids, attention_mask, audio, vision):
        bert_out    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        t           = bert_out.last_hidden_state[:, 0, :]    # (B, 768)
        text_seq    = bert_out.last_hidden_state             # (B, L, 768)

        A = self.audio_enc(audio)   # (B, 50, 768)
        V = self.vision_enc(vision) # (B, 50, 768)

        # Direction 1: text -> audio/vision
        a_att, _ = self.audio_attn(t, A)   # (B, 768)
        v_att, _ = self.vision_attn(t, V)  # (B, 768)

        # Direction 2: audio/vision -> text
        a_query = A.mean(dim=1)  # (B, 768) — audio summary as query
        v_query = V.mean(dim=1)  # (B, 768)
        t_from_a, _ = self.text_attn_from_audio(a_query,  text_seq)  # (B, 768)
        t_from_v, _ = self.text_attn_from_vision(v_query, text_seq)  # (B, 768)

        h = torch.cat([
            t, a_att, v_att, t_from_a, t_from_v,
            t - a_att, t - v_att,
            t_from_a - t, t_from_v - t,
        ], dim=1)  # (B, 6912)

        return self.classifier(h).squeeze(1)

In [27]:
class CMIAResidualV2(nn.Module):
    """
    final_logit = text_logit + incongruity_logit

    Text path: BERT [CLS] -> small MLP -> scalar  (strong text-only anchor)
    Incongruity path: uses stop-gradient on t so BERT optimises for
                      text classification, not for incongruity matching.
    Uses Proj architecture (256-dim) to keep incongruity path small.
    """
    def __init__(self, bert_model_name=BERT_MODEL, audio_input_dim=81,
                 vision_input_dim=371, proj_dim=256,
                 text_hidden=256, inc_hidden=256, dropout=0.3):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(bert_model_name)
        self.text_head  = nn.Sequential(
            nn.Linear(BERT_DIM, text_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(text_hidden, 1),
        )
        self.text_proj   = nn.Linear(BERT_DIM, proj_dim, bias=False)
        self.audio_enc   = ModalityEncoderSmall(audio_input_dim)
        self.vision_enc  = ModalityEncoderSmall(vision_input_dim)
        self.audio_attn  = CrossModalAttentionProj(proj_dim)
        self.vision_attn = CrossModalAttentionProj(proj_dim)
        self.incongruity_head = nn.Sequential(
            nn.Linear(proj_dim * 5, inc_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(inc_hidden, 64),            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, input_ids, attention_mask, audio, vision):
        t          = self.bert(input_ids=input_ids,
                               attention_mask=attention_mask).last_hidden_state[:, 0, :]
        text_logit = self.text_head(t).squeeze(1)

        # Stop gradient: incongruity path cannot update BERT
        t_sg     = t.detach()
        t_proj   = self.text_proj(t_sg)

        A = self.audio_enc(audio)
        V = self.vision_enc(vision)
        a_att, _ = self.audio_attn(t_proj, A)
        v_att, _ = self.vision_attn(t_proj, V)
        h_inc    = torch.cat([t_proj, a_att, v_att,
                               t_proj - a_att, t_proj - v_att], dim=1)
        inc_logit = self.incongruity_head(h_inc).squeeze(1)

        return text_logit + inc_logit

In [33]:
torch.manual_seed(42)
new_results = {}
NUM_EPOCHS = 10

# --- Gated ---
print('=== CMIA-Gated alternating ===')
model_gated = CMIAGatedModel().to(DEVICE)
new_results['CMIA_Gated'] = train_alternating(
    model_gated, train_loader, test_loader,
    num_epochs=NUM_EPOCHS, label='CMIA_Gated')

# --- Bidirectional ---
print('=== CMIA-Bidirectional alternating ===')
model_bidir = CMIABidirectionalModel().to(DEVICE)
new_results['CMIA_Bidir'] = train_alternating(
    model_bidir, train_loader, test_loader,
    num_epochs=NUM_EPOCHS, label='CMIA_Bidir')

# --- Residual V2 ---
print('=== CMIA-Residual-V2 alternating ===')
model_resv2 = CMIAResidualV2().to(DEVICE)
resv2_non_bert = (
    list(model_resv2.text_head.parameters())       +
    list(model_resv2.text_proj.parameters())       +
    list(model_resv2.audio_enc.parameters())       +
    list(model_resv2.vision_enc.parameters())      +
    list(model_resv2.audio_attn.parameters())      +
    list(model_resv2.vision_attn.parameters())     +
    list(model_resv2.incongruity_head.parameters())
)
new_results['CMIA_ResidualV2'] = train_alternating_generic(
    model_resv2, resv2_non_bert, train_loader, test_loader,
    num_epochs=NUM_EPOCHS, label='CMIA_ResV2')

# --- Summary table ---
all_results = {**results, **new_results}
order = [
    'Late Fusion RNN (avg, joint)',
    'CMIA_alternating',
    'CMIA_Proj_alternating',
    'CMIA_Gated',
    'CMIA_Bidir',
    'CMIA_ResidualV2',
]
rows = []
for name in order:
    m = prior.get(name) or all_results.get(name, {})
    rows.append({
        'Model':     name,
        'Test Acc':  f"{m['accuracy']:.4f}"  if m.get('accuracy')  is not None else '-',
        'Precision': f"{m['precision']:.4f}" if m.get('precision') is not None else '-',
        'Recall':    f"{m['recall']:.4f}"    if m.get('recall')    is not None else '-',
        'F1':        f"{m['f1']:.4f}"        if m.get('f1')        is not None else '-',
    })
display(pd.DataFrame(rows))

=== CMIA-Gated alternating ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [CMIA_Gated] Ep 01 (rest) | Train Acc 0.5127 | Test  Acc 0.4638  F1 0.6146
  [CMIA_Gated] Ep 02 (rest) | Train Acc 0.5344 | Test  Acc 0.5725  F1 0.0000
  [CMIA_Gated] Ep 03 (bert) | Train Acc 0.6522 | Test  Acc 0.7101  F1 0.6226
  [CMIA_Gated] Ep 04 (bert) | Train Acc 0.7790 | Test  Acc 0.7246  F1 0.6200
  [CMIA_Gated] Ep 05 (rest) | Train Acc 0.8949 | Test  Acc 0.7101  F1 0.6552
  [CMIA_Gated] Ep 06 (rest) | Train Acc 0.8859 | Test  Acc 0.7464  F1 0.7200
  [CMIA_Gated] Ep 07 (bert) | Train Acc 0.9040 | Test  Acc 0.7246  F1 0.6200
  [CMIA_Gated] Ep 08 (bert) | Train Acc 0.9565 | Test  Acc 0.7464  F1 0.7059
  [CMIA_Gated] Ep 09 (rest) | Train Acc 0.9764 | Test  Acc 0.7246  F1 0.6885
  [CMIA_Gated] Ep 10 (rest) | Train Acc 0.9783 | Test  Acc 0.7246  F1 0.6607
  -> Best F1: 0.7200

=== CMIA-Bidirectional alternating ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

  [CMIA_Bidir] Ep 01 (rest) | Train Acc 0.5562 | Test  Acc 0.5870  F1 0.1231
  [CMIA_Bidir] Ep 02 (rest) | Train Acc 0.6105 | Test  Acc 0.6232  F1 0.6438
  [CMIA_Bidir] Ep 03 (bert) | Train Acc 0.6975 | Test  Acc 0.7174  F1 0.5979
  [CMIA_Bidir] Ep 04 (bert) | Train Acc 0.8098 | Test  Acc 0.7174  F1 0.7310
  [CMIA_Bidir] Ep 05 (rest) | Train Acc 0.8786 | Test  Acc 0.7899  F1 0.7478
  [CMIA_Bidir] Ep 06 (rest) | Train Acc 0.9040 | Test  Acc 0.7681  F1 0.6981
  [CMIA_Bidir] Ep 07 (bert) | Train Acc 0.9004 | Test  Acc 0.7174  F1 0.5714
  [CMIA_Bidir] Ep 08 (bert) | Train Acc 0.9547 | Test  Acc 0.6884  F1 0.4941
  [CMIA_Bidir] Ep 09 (rest) | Train Acc 0.9710 | Test  Acc 0.7464  F1 0.7059
  [CMIA_Bidir] Ep 10 (rest) | Train Acc 0.9692 | Test  Acc 0.7391  F1 0.7049
  -> Best F1: 0.7478

=== CMIA-Residual-V2 alternating ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [CMIA_ResV2] Ep 01 (rest) | Train Acc 0.5507 | Test  Acc 0.6522  F1 0.3333
  [CMIA_ResV2] Ep 02 (rest) | Train Acc 0.6486 | Test  Acc 0.7464  F1 0.7154
  [CMIA_ResV2] Ep 03 (bert) | Train Acc 0.6830 | Test  Acc 0.7609  F1 0.7027
  [CMIA_ResV2] Ep 04 (bert) | Train Acc 0.8424 | Test  Acc 0.7319  F1 0.7218
  [CMIA_ResV2] Ep 05 (rest) | Train Acc 0.9493 | Test  Acc 0.7464  F1 0.6847
  [CMIA_ResV2] Ep 06 (rest) | Train Acc 0.9547 | Test  Acc 0.7464  F1 0.6729
  [CMIA_ResV2] Ep 07 (bert) | Train Acc 0.9420 | Test  Acc 0.7174  F1 0.6829
  [CMIA_ResV2] Ep 08 (bert) | Train Acc 0.9783 | Test  Acc 0.7246  F1 0.6667
  [CMIA_ResV2] Ep 09 (rest) | Train Acc 1.0000 | Test  Acc 0.7246  F1 0.6724
  [CMIA_ResV2] Ep 10 (rest) | Train Acc 1.0000 | Test  Acc 0.7319  F1 0.6408
  -> Best F1: 0.7218



,Model,Test Acc,Precision,Recall,F1
0,"Late Fusion RNN (avg, joint)",0.7899,0.7568,0.7458,0.7434
1,CMIA_alternating,0.7826,0.7544,0.7288,0.7414
2,CMIA_Proj_alternating,0.7391,0.6575,0.8136,0.7273
3,CMIA_Gated,0.7464,0.6818,0.7627,0.7200
4,CMIA_Bidir,0.7899,0.7679,0.7288,0.7478
5,CMIA_ResidualV2,0.7319,0.6486,0.8136,0.7218
